In [2]:
# Install dependencies

%pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

^C
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df = pd.read_csv("/content/sample_data/training_dataset.csv")
if len(df)!=0:
  print("Dataset loaded, total row count:", len(df))

In [ ]:
FEATURES = [
    "college_name",
    "branch_name",
    "category",
    "round",
    "year",
    "quota_type",
    "branch_family",
    "institute_tier",
    "user_percentile",
]

TARGET = "admitted"

X = df[FEATURES]
y = df[TARGET]

categorical_cols = [
    "college_name",
    "branch_name",
    "category",
    "quota_type",
    "branch_family",
]

numeric_cols = [
    "round",
    "year",
    "institute_tier",
    "user_percentile",
]

In [ ]:
preprocessor = ColumnTransformer([
    (
        "cat",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        ),
        categorical_cols
    ),
    (
        "num",
        "passthrough",
        numeric_cols
    )
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=12,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )
}
# training loop:
results = []
for name, model in models.items():
    pipeline = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    print(f"\nTraining: {name}")
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    probs = pipeline.predict_proba(X_test)[:,1]
    accuracy = accuracy_score(y_test, preds)
    roc_auc = roc_auc_score(y_test, probs)
    results.append({
        "Model": name,
        "Accuracy": round(accuracy, 4),
        "ROC_AUC": round(roc_auc, 4)
    })
results_df = pd.DataFrame(results)
print(results_df)